# Le délai compte plus que le nombre d'images

Ce carnet rejoue la section 5.1 de l'article : ce que coûte une
configuration à image unique, ce qu'elle gagne en délai, et ce que vaut la
longueur de la fenêtre de référence.

Trois tables sont employées :

- `exp21_ztest_vs_welch.csv` — l'AUC obtenue par zone, en z-test à une image et
  en test à plusieurs images, avec le délai de la première acquisition ;
- `exp20a_latence_acquisition.csv` — les latences d'acquisition réelles, par
  zone et par aléa ;
- `exp49_fenetre_pre.csv` — le balayage de la longueur de ligne de base, de 3 à
  24 mois.

In [ ]:
import os
import pandas as pd

# les carnets se lancent depuis notebooks/ ; les tables sont dans data/results
RESULTATS = os.path.join("..", "data", "results")
TABLES = os.path.join("..", "tables")

def lire(dossier, nom, **kw):
    """Lit une table du depot. Le separateur et la virgule decimale varient
    d'une campagne a l'autre : on laisse pandas les deviner sauf indication."""
    kw.setdefault("sep", None)
    kw.setdefault("engine", "python")
    return pd.read_csv(os.path.join(dossier, nom), **kw)

def presque(a, b, tol=0.0006):
    """Egalite au millieme pres, tolerance par defaut d'un demi-millieme."""
    assert abs(a - b) < tol, "%.4f attendu, %.4f obtenu" % (b, a)
    return True


Le z-test à une image et le test à plusieurs images, tous cas confondus.

In [ ]:
z = lire(RESULTATS, "exp21_ztest_vs_welch.csv")
print(z.groupby("mode")["auc"].describe()[["count", "mean", "min", "max"]])

Zone par zone : le mode qui l'emporte n'est pas toujours le même.

In [ ]:
# l'AUC contre le delai de la premiere image exploitable, par mode
t = z.pivot_table(index=["pays", "zone"], columns="mode",
                  values=["auc", "delai_premiere_image_h"])
print(t.round(3).to_string())

Nombre moyen d'acquisitions disponibles à 24 h, 48 h et une semaine.

In [ ]:
# ce que la latence reelle autorise
lat = lire(RESULTATS, "exp20a_latence_acquisition.csv")
print(lat.groupby("alea")[["n_images_a_24h", "n_images_a_48h", "n_images_a_168h"]]
      .mean().round(2).to_string())

Le balayage de la fenêtre de référence, et la fenêtre retenue.

In [ ]:
f = lire(RESULTATS, "exp49_fenetre_pre.csv").sort_values("fenetre_pre_mois")
print(f[["fenetre_pre_mois", "auc_mediane", "auc_moyenne", "n_zones",
         "zones_ou_cette_fenetre_est_la_meilleure"]].to_string(index=False))

# l'article retient douze mois comme parametre par defaut : c'est la fenetre
# qui maximise l'AUC mediane sur les zones ou les six fenetres ont abouti.
meilleure = f.loc[f["auc_mediane"].idxmax(), "fenetre_pre_mois"]
print("\nfenetre de meilleure AUC mediane :", meilleure, "mois")